# 🛒 Análise de Vendas — Supermercado (3 Filiais | Jan–Mar 2019)

**Objetivo:** Identificar padrões de receita, comportamento de clientes e desempenho por filial e linha de produto para apoiar decisões comerciais.

**Dataset:** 1.000 transações | 3 filiais (São Paulo, Belo Horizonte, Rio de Janeiro) | 6 linhas de produto

**Ferramentas:** Python · Pandas · Matplotlib · Seaborn

---

## Estrutura da Análise
1. Importação e reconhecimento dos dados  
2. Limpeza e preparação  
3. Desempenho por filial  
4. Desempenho por linha de produto  
5. Perfil de clientes  
6. Métodos de pagamento  
7. Análise temporal (mês e horário de pico)  
8. Avaliação dos clientes  
9. Insights e conclusões  

## 1. Importação e Reconhecimento dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Estilo visual consistente
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold'
})
CORES_FILIAL = {'A': '#2196F3', 'B': '#FF9800', 'C': '#4CAF50'}
PALETA = ['#2196F3', '#FF9800', '#4CAF50', '#E91E63', '#9C27B0', '#00BCD4']

df = pd.read_excel('Vendas_Supermercado.xlsx')
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
print('Tipos de dados:')
print(df.dtypes)
print(f'\nPeríodo: {df["Data"].min().date()} a {df["Data"].max().date()}')
print(f'Filiais: {sorted(df["Filial"].unique())}')
print(f'Linhas de produto: {list(df["Linha de Produto"].unique())}')

## 2. Limpeza e Preparação

In [ ]:
# Verificação de nulos
nulos = df.isnull().sum()
print('Valores nulos por coluna:')
print(nulos[nulos > 0] if nulos.sum() > 0 else '✅ Nenhum valor nulo encontrado.')

# Renomear coluna com espaço extra
df.rename(columns={'Avaliação do Cliente ': 'Avaliação'}, inplace=True)

# Engenharia de features temporais
df['Hora']       = pd.to_datetime(df['Horário'], format='%H:%M:%S').dt.hour
df['DiaSemana']  = df['Data'].dt.day_name()
df['Mes']        = df['Data'].dt.month
df['NomeMes']    = df['Data'].dt.strftime('%b')

# Mapeamento de meses para ordenação
ORDEM_MES = {1: 'Jan', 2: 'Fev', 3: 'Mar'}
df['NomeMes'] = df['Mes'].map(ORDEM_MES)

print('\n✅ Features criadas: Hora, DiaSemana, Mes, NomeMes')
print(f'Receita total do período: R$ {df["Receita Total"].sum():,.2f}')

## 3. Desempenho por Filial

In [ ]:
filial = df.groupby(['Filial', 'Cidade']).agg(
    Receita=('Receita Total', 'sum'),
    Ticket_Medio=('Receita Total', 'mean'),
    Transacoes=('ID', 'count')
).reset_index().sort_values('Receita', ascending=False)

print('Desempenho por Filial:')
filial_fmt = filial.copy()
filial_fmt['Receita']      = filial_fmt['Receita'].map('R$ {:,.2f}'.format)
filial_fmt['Ticket_Medio'] = filial_fmt['Ticket_Medio'].map('R$ {:,.2f}'.format)
print(filial_fmt.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Desempenho por Filial', fontsize=15, fontweight='bold', y=1.01)

metricas = [
    ('Receita', 'Receita Total (R$)', 'Receita por Filial'),
    ('Ticket_Medio', 'Ticket Médio (R$)', 'Ticket Médio por Filial'),
    ('Transacoes', 'Nº de Transações', 'Volume de Transações')
]

for ax, (col, ylabel, titulo) in zip(axes, metricas):
    cores = [CORES_FILIAL[f] for f in filial['Filial']]
    bars = ax.bar(filial['Filial'] + ' - ' + filial['Cidade'],
                  filial[col], color=cores, width=0.5, edgecolor='white')
    ax.set_title(titulo)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('')
    # Rótulos nas barras
    for bar in bars:
        h = bar.get_height()
        label = f'R$ {h:,.0f}' if 'R$' in ylabel else f'{h:,.0f}'
        ax.text(bar.get_x() + bar.get_width()/2, h + h*0.01,
                label, ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.tick_params(axis='x', rotation=10)

plt.tight_layout()
plt.savefig('img_filial.png', bbox_inches='tight')
plt.show()
print('💡 Filial C (Rio de Janeiro) lidera receita total com R$ 110.568, mas por margem pequena.')
print('💡 Filial C também tem o maior ticket médio: R$ 337,10 por transação.')

## 4. Desempenho por Linha de Produto

In [ ]:
produto = df.groupby('Linha de Produto').agg(
    Receita=('Receita Total', 'sum'),
    Ticket=('Receita Total', 'mean'),
    Qtd_Transacoes=('ID', 'count'),
    Avaliacao_Media=('Avaliação', 'mean')
).reset_index().sort_values('Receita', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Receita por linha
bars = axes[0].barh(produto['Linha de Produto'], produto['Receita'],
                    color=PALETA, edgecolor='white')
axes[0].set_title('Receita Total por Linha de Produto')
axes[0].set_xlabel('Receita (R$)')
for bar in bars:
    w = bar.get_width()
    axes[0].text(w + 500, bar.get_y() + bar.get_height()/2,
                 f'R$ {w:,.0f}', va='center', fontsize=9, fontweight='bold')
axes[0].set_xlim(0, produto['Receita'].max() * 1.22)

# Avaliação média por linha
produto_av = produto.sort_values('Avaliacao_Media', ascending=True)
cores_av = ['#E53935' if v < 6.8 else '#43A047' for v in produto_av['Avaliacao_Media']]
bars2 = axes[1].barh(produto_av['Linha de Produto'], produto_av['Avaliacao_Media'],
                     color=cores_av, edgecolor='white')
axes[1].axvline(x=produto_av['Avaliacao_Media'].mean(), color='gray',
                linestyle='--', linewidth=1.2, label=f'Média geral: {produto_av["Avaliacao_Media"].mean():.2f}')
axes[1].set_title('Avaliação Média dos Clientes por Linha')
axes[1].set_xlabel('Nota (1–10)')
axes[1].set_xlim(5.5, 8.5)
axes[1].legend(fontsize=9)
for bar in bars2:
    w = bar.get_width()
    axes[1].text(w + 0.05, bar.get_y() + bar.get_height()/2,
                 f'{w:.2f}', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('img_produto.png', bbox_inches='tight')
plt.show()
print('💡 Alimentos e Bebidas lidera receita (R$ 56.144), mas Saúde e Beleza tem menor faturamento.')
print('💡 Alimentos e Bebidas também tem a melhor avaliação média dos clientes (7,11).')

In [ ]:
# Receita por linha de produto e filial (heatmap)
heatmap_data = df.pivot_table(
    index='Linha de Produto', columns='Filial',
    values='Receita Total', aggfunc='sum'
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(heatmap_data, annot=True, fmt=',.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Receita (R$)'})
ax.set_title('Receita por Linha de Produto × Filial', pad=15)
ax.set_xlabel('Filial')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('img_heatmap.png', bbox_inches='tight')
plt.show()
print('💡 Acessórios Eletrônicos na Filial B e Esportes na Filial C são os maiores destaques individuais.')

## 5. Perfil de Clientes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Perfil de Clientes', fontsize=15, fontweight='bold')

# Receita por tipo de consumidor
tipo = df.groupby('Tipo de Consumidor')['Receita Total'].agg(['sum','mean','count'])
axes[0].bar(tipo.index, tipo['sum'], color=['#5C6BC0','#26A69A'], edgecolor='white', width=0.4)
axes[0].set_title('Receita por Tipo de Consumidor')
axes[0].set_ylabel('Receita Total (R$)')
for i, (idx, row) in enumerate(tipo.iterrows()):
    axes[0].text(i, row['sum'] + 1000, f'R$ {row["sum"]:,.0f}\n({row["count"]} transações)',
                 ha='center', fontsize=9, fontweight='bold')
axes[0].set_ylim(0, tipo['sum'].max() * 1.2)

# Receita por gênero
genero = df.groupby('Gênero')['Receita Total'].sum()
cores_g = ['#E91E63', '#1565C0']
wedges, texts, autotexts = axes[1].pie(
    genero.values, labels=genero.index, autopct='%1.1f%%',
    colors=cores_g, startangle=90, pctdistance=0.75,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts: at.set_fontweight('bold')
axes[1].set_title('Receita por Gênero')

# Produto mais comprado por gênero
genero_prod = df.groupby(['Gênero','Linha de Produto'])['Receita Total'].sum().unstack()
genero_prod.plot(kind='bar', ax=axes[2], color=PALETA, edgecolor='white', width=0.6)
axes[2].set_title('Receita por Linha × Gênero')
axes[2].set_xlabel('')
axes[2].set_ylabel('Receita (R$)')
axes[2].legend(fontsize=7, loc='upper right')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('img_clientes.png', bbox_inches='tight')
plt.show()
print('💡 Membros têm ticket médio R$ 9,67 maior que clientes normais (R$ 327,79 vs R$ 318,12).')
print('💡 Saúde e Beleza é predominantemente comprado por homens; Moda e Casa lideram entre mulheres.')

## 6. Métodos de Pagamento

In [ ]:
pagamento = df.groupby('Método de Pagamento').agg(
    Receita=('Receita Total', 'sum'),
    Transacoes=('ID', 'count'),
    Ticket=('Receita Total', 'mean')
).reset_index().sort_values('Receita', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cores_pag = ['#FF7043','#42A5F5','#66BB6A']

axes[0].bar(pagamento['Método de Pagamento'], pagamento['Receita'],
            color=cores_pag, edgecolor='white', width=0.5)
axes[0].set_title('Receita por Método de Pagamento')
axes[0].set_ylabel('Receita (R$)')
for i, row in pagamento.iterrows():
    axes[0].text(list(pagamento.index).index(i),
                 row['Receita'] + 1000,
                 f'R$ {row["Receita"]:,.0f}\n{row["Transacoes"]} transações',
                 ha='center', fontsize=9, fontweight='bold')
axes[0].set_ylim(0, pagamento['Receita'].max() * 1.2)
axes[0].tick_params(axis='x', rotation=10)

# Pagamento por filial
pag_filial = df.groupby(['Filial','Método de Pagamento'])['Receita Total'].sum().unstack()
pag_filial.plot(kind='bar', ax=axes[1], color=cores_pag, edgecolor='white', width=0.6)
axes[1].set_title('Pagamento por Filial')
axes[1].set_xlabel('Filial')
axes[1].set_ylabel('Receita (R$)')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Método', fontsize=8)

plt.tight_layout()
plt.savefig('img_pagamento.png', bbox_inches='tight')
plt.show()
print('💡 Dinheiro (R$ 112.206) e Carteira Digital (R$ 109.993) superam Cartão de Crédito (R$ 100.767).')
print('💡 Adoção de carteira digital é uniforme entre as filiais — indicativo de perfil digital do cliente.')

## 7. Análise Temporal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Receita por mês
mes = df.groupby(['Mes','NomeMes'])['Receita Total'].sum().reset_index().sort_values('Mes')
bars = axes[0].bar(mes['NomeMes'], mes['Receita Total'],
                   color=['#42A5F5','#EF5350','#66BB6A'], edgecolor='white', width=0.5)
axes[0].set_title('Receita Total por Mês')
axes[0].set_ylabel('Receita (R$)')
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, h + 800,
                 f'R$ {h:,.0f}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_ylim(0, mes['Receita Total'].max() * 1.15)

# Receita por hora do dia
hora = df.groupby('Hora')['Receita Total'].sum().reset_index()
cor_hora = ['#E53935' if h in [13,19] else '#90CAF9' for h in hora['Hora']]
axes[1].bar(hora['Hora'], hora['Receita Total'], color=cor_hora, edgecolor='white')
axes[1].set_title('Receita por Horário do Dia')
axes[1].set_xlabel('Hora')
axes[1].set_ylabel('Receita (R$)')
axes[1].set_xticks(hora['Hora'])
# Anotações nos picos
for _, row in hora[hora['Hora'].isin([13,19])].iterrows():
    axes[1].annotate(f'{int(row["Hora"]):02d}h\nR${row["Receita Total"]/1000:.0f}k',
                     xy=(row['Hora'], row['Receita Total']),
                     xytext=(row['Hora'], row['Receita Total'] + 2500),
                     ha='center', fontsize=9, fontweight='bold', color='#C62828')

plt.tight_layout()
plt.savefig('img_temporal.png', bbox_inches='tight')
plt.show()
print('💡 Janeiro é o mês mais forte (R$ 116.291). Fevereiro cai ~16% — investigar causa.')
print('💡 Dois picos claros de fluxo: 13h (almoço) e 19h (após trabalho). Pico das 19h supera R$ 39k.')

## 8. Avaliação dos Clientes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribuição das avaliações
axes[0].hist(df['Avaliação'], bins=20, color='#5C6BC0', edgecolor='white', alpha=0.85)
media = df['Avaliação'].mean()
axes[0].axvline(media, color='#E53935', linestyle='--', linewidth=2,
                label=f'Média: {media:.2f}')
axes[0].set_title('Distribuição das Avaliações')
axes[0].set_xlabel('Nota (1–10)')
axes[0].set_ylabel('Frequência')
axes[0].legend()

# Avaliação média por filial
av_filial = df.groupby('Filial')['Avaliação'].mean().reset_index()
cores_filial = [CORES_FILIAL[f] for f in av_filial['Filial']]
bars = axes[1].bar(av_filial['Filial'], av_filial['Avaliação'],
                   color=cores_filial, edgecolor='white', width=0.4)
axes[1].set_title('Avaliação Média por Filial')
axes[1].set_ylabel('Nota Média')
axes[1].set_ylim(6.0, 7.8)
axes[1].axhline(df['Avaliação'].mean(), color='gray', linestyle='--',
                linewidth=1.2, label=f'Média geral: {df["Avaliação"].mean():.2f}')
axes[1].legend()
for bar in bars:
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, h + 0.02,
                 f'{h:.2f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('img_avaliacao.png', bbox_inches='tight')
plt.show()
print('💡 Avaliações bem distribuídas (não há concentração em extremos) — sinal de experiência mediana estável.')
print('💡 Filial B (BH) tem a menor avaliação média (6.82). Merece atenção em qualidade de atendimento.')

## 9. Insights e Conclusões

| # | Insight | Recomendação |
|---|---|---|
| 1 | Filial C (Rio) lidera em receita (R$ 110.568) e ticket médio (R$ 337) | Investigar práticas da Filial C para replicar nas demais |
| 2 | Alimentos e Bebidas é a linha mais rentável e mais bem avaliada | Priorizar estoque e promoções nessa categoria |
| 3 | Saúde e Beleza tem o menor faturamento (R$ 49.193) | Avaliar mix de produtos ou estratégia de preços |
| 4 | Janeiro é ~19% mais forte que Fevereiro | Investigar se há sazonalidade ou evento externo |
| 5 | Pico de vendas às 19h (R$ 39.699) — 63% maior que média das demais horas | Reforçar equipe e estoque nesses horários |
| 6 | Filial B tem menor avaliação (6.82) e ticket abaixo das demais | Ação de melhoria de experiência do cliente prioritária |
| 7 | Membros têm ticket médio R$ 9,67 maior que clientes normais | Programa de fidelidade com conversão ativa de clientes normais |

In [ ]:
# Resumo executivo
print('='*55)
print('        RESUMO EXECUTIVO — 3 MESES DE OPERAÇÃO')
print('='*55)
print(f'Receita Total:         R$ {df["Receita Total"].sum():>12,.2f}')
print(f'Total de Transações:        {len(df):>10,}')
print(f'Ticket Médio Geral:    R$ {df["Receita Total"].mean():>12,.2f}')
print(f'Melhor Filial:              {df.groupby("Filial")["Receita Total"].sum().idxmax():>10}')
print(f'Melhor Linha:          {df.groupby("Linha de Produto")["Receita Total"].sum().idxmax():>20}')
print(f'Avaliação Média Geral:      {df["Avaliação"].mean():>10.2f}')
print(f'Horário de Pico:            {df.groupby("Hora")["Receita Total"].sum().idxmax():>9}h')
print('='*55)